In [ ]:
import requests
import pandas as pd
import sqlite3

In [ ]:
url = "https://environment.data.gov.uk/hydrology/id/stations/HIPPER_PARK ROAD BRIDGE_E_202312/measures.json"

response = requests.get(url, timeout=10)
print(response.status_code)
print(response.json())


In [ ]:
import requests

BASE_URL = "https://environment.data.gov.uk/hydrology"

def get_measures_for_station(station_id):
    response = requests.get(
        f"{BASE_URL}/id/measures",
        params={"station": station_id},
        timeout=10
    )
    response.raise_for_status()
    return response.json().get("items", [])

measures = get_measures_for_station("E64999A")

print("Measures found:", len(measures))

for m in measures:
    print(m.get("@id"), "-", m.get("parameter"))


In [ ]:
import requests

# Base API URL and the station ID
BASE_URL = "https://environment.data.gov.uk/hydrology"
STATION_ID = "E64999A"


def station_measures(station_id):
    
    """ Retrieve all measurement types available for a specific station. """
    url = f"{BASE_URL}/id/measures"
    params = {"station": station_id}
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        # items contains the list of measures
        return response.json().get("items", [])
    
    except requests.RequestException as e:
        print(f"Error fetching measures for station {station_id}: {e}")
        return []


def recent_readings(measure_url, limit=10):
    
    """ Fetching the latest readings and limit the number of results to 10. """
    url = f"{measure_url}/readings.json"
    params = {"_limit": limit, "_sort": "-dateTime"}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        return response.json().get("items", [])
    
    except requests.RequestException as e:
        print(f"Error fetching readings from {measure_url}: {e}")
        return []


# Getting all measures for the station
all_measures = station_measures(STATION_ID)

# Filter for the parameters we care about
target_parameters = ["DISSOLVED OXYGEN", "CONDUCTIVITY"]
measurement  = [
    m for m in all_measures if m.get("parameter") in target_parameters
]

# Fetching and display of readings for our desired measure
for measure in measurement :
    parameter_name = measure.get("parameter")
    print(parameter_name)
    
    readings = recent_readings(measure.get("@id"))
    for reading in readings:
        timestamp = reading.get("dateTime")
        value = reading.get("value")
        print(f"{timestamp} | {value}")

rows = []
for measure in measurement:
    readings = recent_readings(measure["@id"])
    for r in readings:
        rows.append({
            "station_id": STATION_ID,
            "parameter": measure.get("parameter"),
            "date_time": r.get("dateTime"),
            "value": r.get("value")
        })


In [ ]:
# Convert to DataFrame
df = pd.DataFrame(rows)
print(df.head())

In [ ]:
df.info()

In [ ]:
# Remove duplicates based on station_id, parameter, and date_time, keeping the last occurrence
df = df.drop_duplicates(subset=["station_id", "parameter", "date_time"], keep="last")

# Convert all column names to lowercase and replace spaces with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_', regex=True)

# Convert all string columns to lowercase and replace spaces with underscores
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.replace(' ', '_', regex=True)
    df[col] = df[col].str.lower()
    
# Convert dateTime to proper datetime type and rename to date_time
df["date_time"] = pd.to_datetime(df["date_time"], errors='coerce')

# Remove rows with missing values
df = df.dropna(subset=["value", "date_time"])

# Ensure value is float
df["value"] = df["value"].astype(float)

# Keep only the latest 10 rows per parameter
df = df.sort_values(["parameter", "date_time"], ascending=[True, False])
df = df.groupby("parameter").head(10).reset_index(drop=True)

print(df)

In [ ]:
# Constants
BASE_URL = "https://environment.data.gov.uk/hydrology"
STATION_ID = "E64999A"
STATION_NAME = "Hipper Park Road Bridge"
TARGET_MEASURES = {
    "DISSOLVED OXYGEN": "mg/l",
    "CONDUCTIVITY": "uS/cm"
}
DB_FILE = "hydrological.db"

#  Format date_time to string for database insertion
df["date_time"] = df["date_time"].dt.strftime("%Y-%m-%d %H:%M:%S")

with sqlite3.connect(DB_FILE) as conn:
    cursor = conn.cursor()

    # Create dimension table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS stations (
        station_id TEXT PRIMARY KEY,
        station_name TEXT
    )
    """)

    cursor.execute("""
    INSERT OR REPLACE INTO stations (station_id, station_name)
    VALUES (?, ?)
    """, (STATION_ID, STATION_NAME))

    # Create fact table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS measurements (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        station_id TEXT,
        parameter TEXT,
        date_time TEXT,
        value REAL,
        UNIQUE(station_id, parameter, date_time),
        FOREIGN KEY(station_id) REFERENCES stations(station_id)
    )
    """)
    conn.commit()
    
    # Convert DataFrame to list of dicts for insertion
    records = df.to_dict(orient="records")
        
    # Using INSERT OR IGNORE to prevent duplicates based on the UNIQUE constraint
    cursor.executemany("""
    INSERT OR IGNORE INTO measurements
    (station_id, parameter, date_time, value)
    VALUES (:station_id, :parameter, :date_time, :value)
    """, records)
        
    conn.commit()

In [ ]:
result = pd.read_sql("SELECT * FROM measurements LIMIT 5", conn)
print(result)